In [3]:
import pandas as pd
import music21 as m21
from fractions import Fraction
from harmonic_inference.data.data_types import ChordType, PitchType, KeyMode
from harmonic_inference.utils.harmonic_utils import get_scale_degree_from_interval, get_pitch_from_string
import harmonic_inference.utils.harmonic_constants as hc
from pathlib import Path
import os

In [4]:
# dicts

chord_types = {
"dominant-seventh": ChordType.MAJ_MIN7,
"major": ChordType.MAJOR,
"major-seventh": ChordType.MAJ_MAJ7,
"diminished": ChordType.DIMINISHED,
"augmented": ChordType.AUGMENTED,
"diminished-seventh": ChordType.DIM7,
"minor": ChordType.MINOR,
"minor-seventh": ChordType.MIN_MIN7,
"half-diminished-seventh": ChordType.HALF_DIM7,
"major-sixth": ChordType.MAJOR, # not defined in ChordType class
"minor-sixth": ChordType.MINOR, # Not defined in ChordType class
}


CHORD_TYPES_TO_STRING_READABLE = {
    ChordType.MAJ_MIN7: "D7",
    ChordType.MAJOR: "M",
    ChordType.MAJ_MAJ7: "M7",
    ChordType.DIMINISHED: "d",
    ChordType.AUGMENTED: "a",
    ChordType.DIM7: "d7",
    ChordType.MINOR: "m",
    ChordType.MIN_MIN7: "m7",
    ChordType.HALF_DIM7: "h7",
}

In [ ]:
def process_score(score, score_id, keys_df):
    """
    Process a music21 Score and extract chord information with keys from CSV.
    
    Args:
        score: music21 Score object

        score_id: score identifier string (e.g., "score_1361")

        keys_df: DataFrame with columns [score_id, key_onset_measure, key, mode]
    
    Returns:
        DataFrame with columns: on, off, key, degree, type, inv
    """
    
    # Get all harmonies in the score
    harmonies = score.flat.getElementsByClass("Harmony")
    
    if not harmonies:
        return pd.DataFrame()
    
    # Load CSV keys for this score
    score_keys = keys_df[keys_df["score_id"] == score_id]
    
    key_changes = []
    
    if not score_keys.empty: # ensure there is key change data in the csv

        # Get actual measures from the score to map measure numbers to offsets
        all_measures = []
        for part in score.parts:
            part_measures = list(part.getElementsByClass(m21.stream.Measure))
            if part_measures:
                all_measures = part_measures
                break  # Use measures from the first part that has them

        measure_by_number = { # this maps each measure.number from the score to its measure object, so key changes can be located by real measure number instead of list position
            measure.number: measure
            for measure in all_measures
            if measure.number is not None
        }

        
        # Build key_changes list from CSV using "actual" measure offsets
        for _, row in score_keys.iterrows():
            measure_num = int(row["key_onset_measure"])
            key_str = row["key"]
            mode = row["mode"]
            
            # Get the offset of this measure from the score
            measure = measure_by_number.get(measure_num)
            if measure is not None:
                offset = measure.offset

            
            # else:
            #     # vibecoded Fallback: try positional lookup, then proportional calculation
            #     measure_index = measure_num - 1
            #     if 0 <= measure_index < len(all_measures):
            #         offset = all_measures[measure_index].offset
            #     else:
            #         highest_harmony_offset = max(h.offset for h in harmonies) # 1. find end of piece
            #         total_measures = len(all_measures) if all_measures else score_keys["key_onset_measure"].max() # 2. Count total measures
            #         quarter_notes_per_measure = highest_harmony_offset / total_measures # 3. Calculate avg measure len
            #         offset = (measure_num - 1) * quarter_notes_per_measure # predict offset
            
            # Create music21 Key object
            major_mode = mode.lower() == "major"
            key_obj = m21.key.Key(key_str, "major" if major_mode else "minor")
            key_changes.append((offset, key_obj))
    
    # else:

    #     # vibecoded Fallback: use XML keys if no CSV keys found
    #     for k in score.flat.getElementsByClass(m21.key.Key):
    #         key_changes.append((k.offset, k))
    #     if not key_changes:
    #         for ks in score.flat.getElementsByClass(m21.key.KeySignature):
    #             key_changes.append((ks.offset, ks.asKey()))
    
    key_changes.sort(key=lambda x: x[0])
    
    # Helper function: get key object at a given offset
    def get_key_obj_at_offset(offset):  
        for ks_offset, k_obj in reversed(key_changes): # reversed() ensures we find the latest active key; iterating forward would always match the very first key change at offset 0.
            if offset >= ks_offset:  
                return k_obj
        return m21.key.Key('C')  # C as default Key object

    # Helper function: format key string
    def format_key_string(k_obj):
        tonic_name = k_obj.tonic.name
        mode = k_obj.mode
        key_str = tonic_name.upper() if mode == "major" else tonic_name.lower()  
        return key_str.replace("-", "b").replace("#", "+")  

    # Helper function: get the measure number for a harmony offset 
    # by checking if first measure in the score is shorter than a normal measure defined by the timeSignature
    pickup_measure_adjustment = 1 if all_measures and getattr(all_measures[0], "barDuration", None) and all_measures[0].duration.quarterLength < all_measures[0].barDuration.quarterLength else 0

    def get_measure_num_at_offset(offset):
        if not all_measures: # fallback
            return None
        for measure_num, measure in reversed(list(enumerate(all_measures, start=1))):
            if offset >= measure.offset:
                return max(measure_num - pickup_measure_adjustment, 0)
        return max(1 - pickup_measure_adjustment, 0) # fallback

    # Get score length (in quarter notes)
    piece_len = score.flat.highestOffset

    rows = []

    for harmony in harmonies: # = score.flat.getElementsByClass("Harmony")

        on = Fraction(harmony.offset)
        measure_num = get_measure_num_at_offset(on)

        # Get the correct key at this harmony's offset
        
        current_key_obj = get_key_obj_at_offset(on)
        key_str = format_key_string(current_key_obj)
        key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

        # Extract chord root

        if hasattr(harmony, "root") and harmony.root() is not None:
            chord_root_string = harmony.root().name.replace("-", "b")
        else:
            chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

        # Calculate roman numeral degree

        chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC)
        key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
        degree = get_scale_degree_from_interval(
            chord_root_int - key_tonic_int,
            key_mode,
            PitchType.TPC,
        )

        harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
        chord_type_enum = chord_types.get(harmony_type_str, None)

        if chord_type_enum is not None:
            chord_type = CHORD_TYPES_TO_STRING_READABLE[chord_type_enum]

        else:
            chord_type = "unknown"

        inv = 0

        rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there # / ADD "measure_num" for debug

    # Create DataFrame WITHOUT column names

    df = pd.DataFrame(rows, columns=["on", "off", "key", "degree", "type", "inv"]) # / ADD "measure_num" for debug

    # Shift the 'on' time of the next chord to become the "off" time of the current chord
    df["off"] = df["on"].shift(-1)

    # Handle the last chord's "off" time

    if not df.empty:
        df.iloc[-1, df.columns.get_loc("off")] = Fraction(piece_len)

    return df


In [ ]:
# Load keys and test on a single score
keys_df = pd.read_csv("choro_corpus/keys_choro_pieces.csv")
test_score = m21.converter.parse("choro_corpus/with_chords/score_265-Paixao_encoberta-Mario_Alvares.xml")
process_score(test_score, "score_265", keys_df)


In [4]:
# NON DEBUG
def process_score(score, score_id, keys_df):
    """
    Process a music21 Score and extract chord information with keys from CSV.
    
    Args:
        score: music21 Score object

        score_id: score identifier string (e.g., "score_1361")

        keys_df: DataFrame with columns [score_id, key_onset_measure, key, mode]
    
    Returns:
        DataFrame with columns: on, off, key, degree, type, inv
    """
    
    # Get all harmonies in the score
    harmonies = score.flat.getElementsByClass("Harmony")
    
    if not harmonies:
        return pd.DataFrame()
    
    # Load CSV keys for this score
    score_keys = keys_df[keys_df["score_id"] == score_id]
    
    key_changes = []
    
    if not score_keys.empty: # ensure there is key change data in the csv

        # Get actual measures from the score to map measure numbers to offsets
        all_measures = []
        for part in score.parts:
            part_measures = part.getElementsByClass(m21.stream.Measure)
            if part_measures:
                all_measures = list(part_measures)
                break  # Use measures from the first part that has them
        
        # Build key_changes list from CSV using actual measure offsets
        for _, row in score_keys.iterrows():
            measure_num = row["key_onset_measure"]
            key_str = row["key"]
            mode = row["mode"]
            
            # Get the actual offset of this measure from the score
            if measure_num <= len(all_measures):
                measure = all_measures[measure_num - 1] # convert 1-based measure number from csv to 0-based python index
                offset = measure.offset
            else:
                # Fallback: if measure doesn't exist, use proportional calculation
                highest_harmony_offset = max(h.offset for h in harmonies) # 1. find end of piece
                total_measures = len(all_measures) if all_measures else score_keys["key_onset_measure"].max() # 2. Count total measures
                quarter_notes_per_measure = highest_harmony_offset / total_measures # 3. Calculate avg measure len
                offset = (measure_num - 1) * quarter_notes_per_measure # predict offset
            
            # Create music21 Key object
            major_mode = mode.lower() == "major"
            key_obj = m21.key.Key(key_str, "major" if major_mode else "minor")
            key_changes.append((offset, key_obj))
    else:

        # Fallback: use XML keys if no CSV keys found
        for k in score.flat.getElementsByClass(m21.key.Key):
            key_changes.append((k.offset, k))
        if not key_changes:
            for ks in score.flat.getElementsByClass(m21.key.KeySignature):
                key_changes.append((ks.offset, ks.asKey()))
    
    key_changes.sort(key=lambda x: x[0])
    
    # Helper function: get key object at a given offset
    def get_key_obj_at_offset(offset):  
        for ks_offset, k_obj in reversed(key_changes):  
            if offset >= ks_offset:  
                return k_obj
        return m21.key.Key('C')  # C as default Key object

    # Helper function: format key string
    def format_key_string(k_obj):
        tonic_name = k_obj.tonic.name
        mode = k_obj.mode
        key_str = tonic_name.upper() if mode == "major" else tonic_name.lower()  
        return key_str.replace("-", "b").replace("#", "+")  

    # Get score length (in quarter notes)
    piece_len = score.flat.highestOffset

    rows = []

    for harmony in harmonies: # score.flat.getElementsByClass("Harmony")

        on = Fraction(harmony.offset)

        # Get the correct key at this harmony's offset
        
        current_key_obj = get_key_obj_at_offset(on)
        key_str = format_key_string(current_key_obj)
        key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

        # Extract chord root

        if hasattr(harmony, "root") and harmony.root() is not None:
            chord_root_string = harmony.root().name.replace("-", "b")
        else:
            chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

        # Calculate roman numeral degree

        chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC)
        key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
        degree = get_scale_degree_from_interval(
            chord_root_int - key_tonic_int,
            key_mode,
            PitchType.TPC,
        )

        harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
        chord_type_enum = chord_types.get(harmony_type_str, None)

        if chord_type_enum is not None:
            chord_type = CHORD_TYPES_TO_STRING_READABLE[chord_type_enum]

        else:
            chord_type = "unknown"

        inv = 0

        rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there

    # Create DataFrame WITHOUT column names

    df = pd.DataFrame(rows) # columns = ["on", "off", "key", "degree", "type", "inv"]

    # Shift the 'on' time of the next chord to become the "off" time of the current chord
    df[1] = df[0].shift(-1)

    # Handle the last chord's "off" time

    if not df.empty:
        df.iloc[-1, 1] = Fraction(piece_len)

    return df


In [17]:
# Load keys from CSV
keys_df = pd.read_csv("choro_corpus/keys_choro_pieces.csv")

# Process entire directory
input_dir = Path("choro_corpus/with_chords")
output_dir = Path("choro_model_ft/chords")

output_dir.mkdir(exist_ok=True)

for file in input_dir.glob("*.xml"):

    try:
        score = m21.converter.parse(file)
        
        # Extract score_id from filename (everything before first dash)
        score_id = file.stem.split('-')[0]

        df = process_score(score, score_id, keys_df)

        output_file = output_dir / f"{file.stem}.csv"

        df.to_csv(
            output_file,
            index=False,
            header=False
        )
        print(f"✓ Processed {file.stem}")
    except Exception as e: 
        print(f"✗ Failed on {file.name}: {e}")


✓ Processed score_4566-Isto_nao_e_vida-Macarico
✓ Processed score_1985-Tira_Poeira-Satyro_Bilhar
✓ Processed score_1361-Medrosa-Anacleto_de_Medeiros
✓ Processed score_265-Paixao_encoberta-Mario_Alvares
✗ Failed on score_2802-Belezas_do_Recife-Misael_Domingues.xml: 'N.C.'
✓ Processed score_5063-Pinguim-Ernesto_Nazareth
✓ Processed score_5789-Tristeza-Eduardo_Souto
✓ Processed score_117-Bohemia_Terra-Irineu_de_Almeida
✓ Processed score_5602-Flor_do_Abacate-Alvaro_Sandim
✓ Processed score_4990-Menino_de_Ouro-Ernesto_Nazareth
✓ Processed score_2622-Tu_passaste_por_este_Jardim-Alfredo_Dutra
✓ Processed score_11143-Gemea-Chiquinha_Gonzaga
✓ Processed score_192-Qualquer_coisa-Irineu_de_Almeida
✓ Processed score_12248-Gloria-Bonfiglio_de_Oliveira
✓ Processed score_10805-Meu_sabia-Raul_Silva
✓ Processed score_11265-Por_um_beijo_Terna_saudade-Anacleto_de_Medeiros


# Everything below are older versions / tests!

In [ ]:
# TEST: compare score_265 and score_1361 after measure-number based key mapping
keys_df = pd.read_csv("choro_corpus/keys_choro_pieces.csv")

for score_id, xml_name, boundary_measures in [
    ("score_265", "choro_corpus/with_chords/score_265-Paixao_encoberta-Mario_Alvares.xml", [33, 65]),
    ("score_1361", "choro_corpus/with_chords/score_1361-Medrosa-Anacleto_de_Medeiros.xml", [17, 26]),
]:
    score = m21.converter.parse(xml_name)
    df = process_score(score, score_id, keys_df)
    print(f"\n{score_id} key distribution:")
    print(df["key"].value_counts().sort_index())
    print("boundary rows:")
    for measure_num in boundary_measures:
        window = df[(df["measure_num"] >= measure_num - 1) & (df["measure_num"] <= measure_num + 1)]
        print(f"  around measure {measure_num}:")
        print(window[["measure_num", "on", "key", "degree"]].to_string(index=False))


In [ ]:
# TEST: Process score_1361 with updated function and check key distribution
keys_df = pd.read_csv("keys_choro_pieces.csv")
score = m21.converter.parse("choro_corpus/with_chords/score_1361-Medrosa-Anacleto_de_Medeiros.xml")
df = process_score(score, "score_1361", keys_df)

print("Score 1361 - Key Signature Mapping (UPDATED):")
print(f"Unique keys: {sorted(df[2].unique())}")
print(f"\nKey distribution:")
print(df[2].value_counts().sort_index())

print(f"\nRows where Bb major starts (offset >= 49.25):")
bb_rows = df[df[0] >= 49.25]
print(f"Number of Bb major rows: {len(bb_rows)}")
print(f"\nFirst 3 Bb major rows:")
print(bb_rows[[0, 1, 2, 3]].head(3).to_string(index=False))


Score 1361 - Key Signature Mapping (UPDATED):
Unique keys: ['Bb', 'D', 'd']

Key distribution:
2
Bb    23
D     16
d     18
Name: count, dtype: int64

Rows where Bb major starts (offset >= 49.25):
Number of Bb major rows: 23

First 3 Bb major rows:
    0     1  2   3
197/4 201/4 Bb III
201/4 205/4 Bb   V
205/4 213/4 Bb   I


In [ ]:

key_changes = [] 
   
for k in score.flat.getElementsByClass(m21.key.Key):  # actual key (major / minor)

    key_changes.append((k.offset, k))  
if not key_changes:
    for ks in score.flat.getElementsByClass(m21.key.KeySignature): # count accidentals 

        key_changes.append((ks.offset, ks.asKey()))  # score.analyze("key")??



key_changes.sort(key=lambda x: x[0])  

# return the actual music21 Key object, not just the string
def get_key_obj_at_offset(offset):  
    for ks_offset, k_obj in reversed(key_changes):  
        if offset >= ks_offset:  
            return k_obj
    return m21.key.Key('C')  # C as default Key object

# format key str
def format_key_string(k_obj):
    tonic_name = k_obj.tonic.name
    mode = k_obj.mode
    key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
    return key_str.replace('-', 'b').replace('#', '+')  

# get offset length in score (in quarter note length) 

piece_len = score.flat.highestOffset


rows = []

for harmony in score.flat.getElementsByClass("Harmony"):

    on = Fraction(harmony.offset)

    current_key_obj = get_key_obj_at_offset(on)
    key_str = format_key_string(current_key_obj)
    key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

    if hasattr(harmony, "root") and harmony.root() is not None:
        chord_root_string = harmony.root().name.replace("-", "b")
    else:
        chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

    chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC) # not replacing + or - with # and b will make the code crash
    key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
    degree = get_scale_degree_from_interval(
        chord_root_int - key_tonic_int,
        key_mode,
        PitchType.TPC,
    )

    harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
    chord_type = chord_types.get(harmony_type_str, "unknown")

    chord_type = str(chord_type) # make it a string


    inv = 0

    rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there

# Create DataFrame WITHOUT column names

df = pd.DataFrame(rows) # columns = ["on", "off", "key", "degree", "type", "inv"]

# Shift the 'on' time of the next chord to become the "off" time of the current chord
df[1] = df[0].shift(-1)

# Handle the last chord's "off" time using its original duration since shift leaves None

if not df.empty: # ensures the code only runs if score contains chords
    last_harmony = score.flat.getElementsByClass("Harmony")[-1]

    df.iloc[-1, 1] = Fraction(piece_len) # last_harmony.duration.quarterLength

df




In [ ]:
# BACKUP

chord_types = {
"dominant-seventh": ChordType.MAJ_MIN7,
"major": ChordType.MAJOR,
"major-seventh": ChordType.MAJ_MAJ7,
"diminished": ChordType.DIMINISHED,
"augmented": ChordType.AUGMENTED,
"diminshed-seventh": ChordType.DIM7,
"minor": ChordType.MINOR,
"minor-seventh": ChordType.MIN_MIN7,
"Gr+6": ChordType.MAJOR, # ?
"Fr+6": ChordType.MAJOR, # ?
"It+6": ChordType.MAJOR, # ?
"half-diminished-seventh": ChordType.HALF_DIM7,
}

key_changes = [] 


for k in score.flat.getElementsByClass(m21.key.Key):  
    key_changes.append((k.offset, k))  

if not key_changes:
    for ks in score.flat.getElementsByClass(m21.key.KeySignature):
        key_changes.append((ks.offset, ks.asKey()))  
  
key_changes.sort(key=lambda x: x[0])  

# return the actual music21 Key object, not just the string
def get_key_obj_at_offset(offset):  
    for ks_offset, k_obj in reversed(key_changes):  
        if offset >= ks_offset:  
            return k_obj
    return m21.key.Key('C')  # C as default Key object

# format key str
def format_key_string(k_obj):
    tonic_name = k_obj.tonic.name
    mode = k_obj.mode
    key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
    return key_str.replace('-', 'b').replace('#', '+')  


rows = []

for harmony in score.flat.getElementsByClass("Harmony"):  

    # get chord onset position
    on = Fraction(harmony.offset)
    
    # 1. Get the music21 Key object for this offset
    current_key_obj = get_key_obj_at_offset(on)
    key_str = format_key_string(current_key_obj)
    
    # 2. Calculate the Roman Numeral Degree

    chord_figure = harmony.figure if harmony.figure else str(harmony)
    degree = "unk" # TBD
    inv = 0 # for now ! 
    
    # 3. Get Chord Type mapping
    # harmony.chordKind gives us things like 'major', 'major-seventh', etc.
    harmony_type_str = harmony.chordKind or "major" # default to major ?

    chord_type = chord_types.get(harmony_type_str, "unknown")

    # Append all details to rows (leaving 'off' to be filled by the shift) 
    rows.append([on, None, key_str, degree, chord_type, inv])

# Create DataFrame with explicit column names (IMPORTANT: COLUMN NAMES NEED TO BE DELETED LATER!)
df = pd.DataFrame(rows, columns=["on", "off", "key", "degree", "type", "inv"])

# Shift the 'on' time of the next chord to become the 'off' time of the current chord
df["off"] = df["on"].shift(-1)

# Handle the last chord's 'off' time using its original duration since shift leaves None
if not df.empty:
    last_harmony = score.flat.getElementsByClass("Harmony")[-1]
    df.iloc[-1, df.columns.get_loc("off")] = Fraction(last_harmony.offset + last_harmony.duration.quarterLength)

# df.to_csv("score4990example.csv")

df

In [ ]:
# TESTS


# labels_df = pd.read_csv(
#         label_csv_path,
#         header=None,
#         names=["on", "off", "key", "degree", "type", "inv"],
#         dtype={"degree": str},
#         converters={"on": Fraction, "off": Fraction},
# )
chord_types = {
"D7": ChordType.MAJ_MIN7,
"M": ChordType.MAJOR,
"M7": ChordType.MAJ_MAJ7,
"d": ChordType.DIMINISHED,
"a": ChordType.AUGMENTED,
"d7": ChordType.DIM7,
"m": ChordType.MINOR,
"m7": ChordType.MIN_MIN7,
"Gr+6": ChordType.MAJOR,
"Fr+6": ChordType.MAJOR,
"It+6": ChordType.MAJOR,
"h7": ChordType.HALF_DIM7,
}

# example score
score = m21.converter.parse("choro_corpus/with_chords/score_4990-Menino_de_Ouro-Ernesto_Nazareth.xml")

key_changes = []  

# .flat handles the timeline alignment across all parts safely
for k in score.flat.getElementsByClass(m21.key.Key):  
    key_changes.append((k.offset, k))  

if not key_changes:
    for ks in score.flat.getElementsByClass(m21.key.KeySignature):
        # .asKey() forces '1 flat' to become an F Major Key object
        key_changes.append((ks.offset, ks.asKey()))  
  
# sort by offset  
key_changes.sort(key=lambda x: x[0])  
  
# Function to get key at a given offset  
def get_key_at_offset(offset):  
    for ks_offset, k_obj in reversed(key_changes):  
        if offset >= ks_offset:  
            # k_obj is now a Key object, so it has a .tonic (Note) and .mode ('major'/'minor')
            tonic_name = k_obj.tonic.name # e.g., 'C', 'F', 'G#'
            mode = k_obj.mode
            # Convert to required format: upper case for major, lower for minor  
            key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
            
            # replace flats/sharps with b/+  
            key_str = key_str.replace('-', 'b').replace('#', '+')  
            return key_str  
            
    return "C"  # default

rows = []  
# Using .flat here ensures the Harmony offsets match the Key offsets 
for harmony in score.flat.getElementsByClass("Harmony"):  
    on_before = None
    on = Fraction(harmony.offset)
    off = Fraction(on + harmony.duration.quarterLength)


    key = get_key_at_offset(on)
    rows.append([on, off, key])

df = pd.DataFrame(rows)

# shift to get chord offset position 
df[1] = df[0].shift(-1)

df
# Disambiguate between major and minor (A-major; fsharp-minor, etc.)
# TBD: "degree", "type", "inv"

